# Violent contact classifier (owner: P2)
Frozen MobileNetV2 features (per frame) -> LSTM over 16 frames -> one probability.

**Why this notebook is built the way it is**
- Main data: Real Life Violence Situations (Kaggle), because RWF-2000 needs an agreement with its authors. Same idea: adult fights vs normal actions.
- That footage is adults on the street / CCTV. Our target is children at home. So we do not trust the RWF score alone:
  we add hard negatives (hug, high five, handshake) and we pick thresholds on **our own staged clips**.
- 16 frames taken from (at most) the middle 5.3 s of each clip = **3 fps**, the same rate the backend analyses at. Keep it that way.
- Only the LSTM head trains, so it is fast on Colab. Features are cached once.

Run on Colab with a GPU. Never commit datasets, clips or `violence.pt`.

## 0. Setup

In [ ]:
import os, glob, random, json
import numpy as np
import cv2, torch, torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 16          # frames per clip
IMG = 224             # resize
FEAT_DIM = 1280       # MobileNetV2 output

# Edit these to where the data lives (Drive on Colab). Never inside the git repo.
DATA = '/content/drive/MyDrive/child_day_data'
RWF_DIR    = f'{DATA}/violence_data'  # train|val / Fight|NonFight / video files (built by section 0b; RWF-2000 can be dropped here too)
EXTRA_DIR  = f'{DATA}/extra'        # fight|nofight / *.avi|mp4  (HMDB51 punch/kick; hug, high five, handshake...)
STAGED_DIR = f'{DATA}/staged'       # <scene_name>/*.mp4  e.g. push, hug, high_five, play, calm
CACHE_DIR  = f'{DATA}/feature_cache'
OUT_PATH   = f'{DATA}/violence.pt'  # on Drive so it survives the Colab session; copy to backend/weights/ by hand (not committed)
print(DEVICE)

In [ ]:
# Colab only: mount Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

# Only after the mount (creating this folder before it would break the mount).
os.makedirs(CACHE_DIR, exist_ok=True)


## 0b. Get the data (run once, on Colab, straight into Drive)
Nothing is downloaded to your own computer. Two sources:
1. **Real Life Violence Situations** from Kaggle (`mohamedmustafa/real-life-violence-situations-dataset`), split 80/20 into `train/` and `val/`.
   Used because RWF-2000 needs an agreement with its authors. If you later get RWF-2000, put it in the same `train|val/Fight|NonFight` layout.
2. **HMDB51** from the Hugging Face mirror `CVML-TueAI/HMDB51` (the old Brown University link is dead). Only these classes:
   - violent: `punch`, `kick`, `hit`
   - hard negatives (not violent): touching (`hug`, `shake_hands`, `clap`, `wave`, `kiss`) and fast motion
     (`cartwheel`, `somersault`, `handstand`, `flic_flac`, `jump`, `run`, `kick_ball`, `catch`, `throw`, `dribble`, `swing_baseball`, `fall_floor`).
     The fast-motion ones were added because the first model's false alarms on val were sport and dance clips.

Safe to re-run: only missing classes are downloaded.

In [ ]:
!pip -q install kagglehub huggingface_hub

In [ ]:
import shutil, kagglehub
VIDEO_EXT = ('.avi', '.mp4', '.mov', '.mkv')

def videos_under(folder):
    return [os.path.join(r, f) for r, _, fs in os.walk(folder) for f in fs if f.lower().endswith(VIDEO_EXT)]

if not os.path.isdir(f'{RWF_DIR}/train/Fight'):
    src = kagglehub.dataset_download('mohamedmustafa/real-life-violence-situations-dataset')
    print('downloaded to', src)
    fight, nofight = [], []
    for r, dirs, files in os.walk(src):
        vids = [os.path.join(r, f) for f in files if f.lower().endswith(VIDEO_EXT)]
        if not vids:
            continue
        name = os.path.basename(r).lower().replace('_', '').replace('-', '').replace(' ', '')
        (nofight if 'nonviolen' in name else fight if 'violen' in name else []).extend(vids)
    # Same clip can appear twice in nested copies of the dataset: keep one per file name,
    # otherwise a clip can land in both train and val.
    fight = sorted({os.path.basename(v).lower(): v for v in fight}.values())
    nofight = sorted({os.path.basename(v).lower(): v for v in nofight}.values())
    print('violent clips:', len(fight), '| non-violent clips:', len(nofight))
    assert fight and nofight, 'Could not find Violence / NonViolence folders: look at the printed path and adjust.'
    random.Random(SEED).shuffle(fight); random.Random(SEED).shuffle(nofight)
    for label, vids in (('Fight', fight), ('NonFight', nofight)):
        cut = int(len(vids) * 0.8)
        for split, part in (('train', vids[:cut]), ('val', vids[cut:])):
            os.makedirs(f'{RWF_DIR}/{split}/{label}', exist_ok=True)
            for v in part:
                shutil.copy(v, f'{RWF_DIR}/{split}/{label}/{os.path.basename(v)}')
    print('done ->', RWF_DIR)
else:
    print('violence_data already on Drive, skipping')

In [ ]:
from huggingface_hub import snapshot_download

FIGHT = ['punch', 'kick', 'hit']
# Hard negatives: touching (hug...) AND fast non-violent motion (sport, acrobatics), the val false alarms were sport and dance.
NOFIGHT = ['hug', 'shake_hands', 'clap', 'wave', 'kiss',
           'cartwheel', 'somersault', 'handstand', 'flic_flac', 'jump', 'run',
           'kick_ball', 'catch', 'throw', 'dribble', 'swing_baseball', 'fall_floor']
CLASS_MAP = {**{c: 'fight' for c in FIGHT}, **{c: 'nofight' for c in NOFIGHT}}

existing = {t: os.listdir(f'{EXTRA_DIR}/{t}') if os.path.isdir(f'{EXTRA_DIR}/{t}') else [] for t in ('fight', 'nofight')}
missing = [c for c, t in CLASS_MAP.items() if not any(f.startswith(c + '_') for f in existing[t])]
print('classes to download:', missing or 'none')
if missing:
    tmp = snapshot_download(repo_id='CVML-TueAI/HMDB51', repo_type='dataset',
                            allow_patterns=[f'hmdb51_org/{c}/*' for c in missing])
    for cls in missing:
        target = CLASS_MAP[cls]
        os.makedirs(f'{EXTRA_DIR}/{target}', exist_ok=True)
        for v in videos_under(f'{tmp}/hmdb51_org/{cls}'):
            shutil.copy(v, f'{EXTRA_DIR}/{target}/{cls}_{os.path.basename(v)}')
for t in ('fight', 'nofight'):
    print(t, len(videos_under(f'{EXTRA_DIR}/{t}')))

## 1. Data
Expected layout on Drive (`child_day_data/`):
```
violence_data/{train,val}/{Fight,NonFight}/*   built by 0b (or RWF-2000 in the same layout)
extra/fight/*      HMDB51 punch, kick, hit              (train only)
extra/nofight/*    HMDB51 hug, handshake, clap... + sport/acrobatics (hard negatives, train only)
staged/<scene>/*   our own clips. Folder name containing push, hit, kick, grab, slap or punch = violent; anything else = not violent
```
- Extra clips go to **train only**, so the validation numbers stay comparable.
- Staged clips are **never trained on**. They are the honest test and the place to pick thresholds.
- The notebook still runs with no staged clips yet; section 5 just stays empty.

In [ ]:
VIDEO_EXT = ('.avi', '.mp4', '.mov', '.mkv')
VIOLENT_SCENES = ('push', 'hit', 'kick', 'grab', 'slap', 'punch')

def list_videos(folder):
    out = []
    for ext in VIDEO_EXT:
        out += glob.glob(os.path.join(folder, '**', f'*{ext}'), recursive=True)
    return sorted(out)

def build_items():
    train, val, staged = [], [], []
    for split, bucket in (('train', train), ('val', val)):
        for cls, y in (('Fight', 1), ('NonFight', 0)):
            for p in list_videos(f'{RWF_DIR}/{split}/{cls}'):
                bucket.append((p, y, 'rwf'))
    for cls, y in (('fight', 1), ('nofight', 0)):
        for p in list_videos(f'{EXTRA_DIR}/{cls}'):
            train.append((p, y, 'extra'))
    if os.path.isdir(STAGED_DIR):
        for scene in sorted(os.listdir(STAGED_DIR)):
            y = int(any(k in scene.lower() for k in VIOLENT_SCENES))
            for p in list_videos(f'{STAGED_DIR}/{scene}'):
                staged.append((p, y, scene))
    return train, val, staged

train_items, val_items, staged_items = build_items()
print(len(train_items), 'train |', len(val_items), 'val |', len(staged_items), 'staged')

## 2. Frame sampling and cached features

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
ANALYZED_FPS = 3.0                      # the backend analyses video at 3 fps
WINDOW_S = SEQ_LEN / ANALYZED_FPS       # 16 frames at 3 fps = 5.3 s, same as the Engine will see
MAX_READ = 1800                         # safety cap on frames read from one file

def sample_frames(path, n=SEQ_LEN):
    """n RGB frames (IMG x IMG) covering at most WINDOW_S seconds from the middle of the clip.
    Short clips are sampled evenly over their whole length. Returns None if the video can't be read."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    if not (1 <= fps <= 240):
        fps = 30.0
    frames = []
    while len(frames) < MAX_READ:
        ok, f = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(cv2.resize(f, (IMG, IMG)), cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        return None
    win = int(round(WINDOW_S * fps))
    if len(frames) > win:                       # long clip: take the middle WINDOW_S seconds
        start = (len(frames) - win) // 2
        frames = frames[start:start + win]
    idxs = np.linspace(0, len(frames) - 1, n).astype(int)
    return [frames[k] for k in idxs]

def preprocess(frames):
    x = (np.stack(frames).astype(np.float32) / 255.0 - MEAN) / STD
    return torch.from_numpy(x).permute(0, 3, 1, 2)   # (T, 3, H, W)

backbone = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1).features
backbone = nn.Sequential(backbone, nn.AdaptiveAvgPool2d(1), nn.Flatten()).to(DEVICE).eval()
for p in backbone.parameters():
    p.requires_grad = False     # frozen: only the LSTM head trains

@torch.no_grad()
def extract(frames):
    return backbone(preprocess(frames).to(DEVICE)).cpu().numpy()   # (T, 1280)

def cached_features(path, flip=False):
    """(SEQ_LEN, 1280) features, cached as .npy on Drive. None if the video can't be read."""
    key = os.path.relpath(path, DATA).replace(os.sep, '_').replace('/', '_') + ('_flip' if flip else '') + '.npy'
    fp = os.path.join(CACHE_DIR, key)
    if os.path.exists(fp):
        return np.load(fp)
    frames = sample_frames(path)
    if frames is None:
        return None
    if flip:
        frames = [f[:, ::-1].copy() for f in frames]
    feats = extract(frames)
    np.save(fp, feats)
    return feats

In [ ]:
from tqdm.auto import tqdm
# One pass over everything (the slow step). Cached on Drive, so a second run is fast.
def cache_all(items, flip_too):
    kept, bad = [], []
    for item in tqdm(items):
        if cached_features(item[0]) is None:
            bad.append(item[0]); continue
        if flip_too:
            cached_features(item[0], flip=True)   # free augmentation: mirrored clip
        kept.append(item)
    if bad:
        print(f'skipped {len(bad)} unreadable videos, e.g. {bad[:3]}')
    return kept

train_items  = cache_all(train_items, flip_too=True)
val_items    = cache_all(val_items, flip_too=False)
staged_items = cache_all(staged_items, flip_too=False)
print(len(train_items), 'train |', len(val_items), 'val |', len(staged_items), 'staged')
assert train_items and val_items, 'No training/validation clips: run section 0b first.'

## 3. Model

In [ ]:
class ViolenceHead(nn.Module):
    """LSTM over per-frame MobileNetV2 features -> one logit."""
    def __init__(self, feat_dim=FEAT_DIM, hidden=256, drop=0.5):
        super().__init__()
        self.lstm = nn.LSTM(feat_dim, hidden, batch_first=True)
        self.drop = nn.Dropout(drop)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):                 # x: (B, T, 1280)
        out, _ = self.lstm(x)
        return self.fc(self.drop(out[:, -1])).squeeze(-1)

class FeatDS(Dataset):
    """Features are loaded into memory once (fast epochs)."""
    def __init__(self, items, augment=False):
        self.augment = augment
        self.y = [float(y) for _, y, _ in items]
        self.x = [torch.from_numpy(cached_features(p)).float() for p, _, _ in items]
        self.xf = [torch.from_numpy(cached_features(p, flip=True)).float() for p, _, _ in items] if augment else None
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.xf[i] if (self.augment and random.random() < 0.5) else self.x[i]
        if self.augment:
            x = x + 0.01 * torch.randn_like(x)
        return x, torch.tensor(self.y[i])

train_dl = DataLoader(FeatDS(train_items, augment=True), batch_size=32, shuffle=True)
val_dl   = DataLoader(FeatDS(val_items), batch_size=64)

model = ViolenceHead().to(DEVICE)
n_pos = sum(y for _, y, _ in train_items); n_neg = len(train_items) - n_pos
pos_weight = torch.tensor(n_neg / max(n_pos, 1), device=DEVICE)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
print('positives', n_pos, 'negatives', n_neg)

## 4. Train and evaluate

In [ ]:
@torch.no_grad()
def predict(dl):
    model.eval(); ps, ys = [], []
    for x, y in dl:
        ps += torch.sigmoid(model(x.to(DEVICE))).cpu().tolist(); ys += y.tolist()
    return np.array(ps), np.array(ys)

best_f1, best_state, patience, bad = -1.0, None, 8, 0
history = []
for epoch in range(60):
    model.train(); tot = 0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); loss = loss_fn(model(x), y); loss.backward(); opt.step()
        tot += loss.item() * len(x)
    ps, ys = predict(val_dl)
    f1 = f1_score(ys, ps > 0.5, zero_division=0)
    history.append({'epoch': epoch, 'loss': tot / len(train_items), 'val_acc': float(accuracy_score(ys, ps > 0.5)), 'val_f1': float(f1)})
    print(f'epoch {epoch:02d} loss {tot/len(train_items):.4f} val acc {accuracy_score(ys, ps>0.5):.3f} f1 {f1:.3f}')
    if f1 > best_f1:
        best_f1, bad = f1, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= patience: break
model.load_state_dict(best_state)
print('best val F1', round(best_f1, 3))

In [ ]:
import matplotlib.pyplot as plt
ps, ys = predict(val_dl); pred = ps > 0.5
print('accuracy ', accuracy_score(ys, pred))
print('precision', precision_score(ys, pred, zero_division=0))
print('recall   ', recall_score(ys, pred, zero_division=0))
print('F1       ', f1_score(ys, pred, zero_division=0))
cm = confusion_matrix(ys, pred)
plt.imshow(cm, cmap='Blues'); plt.xticks([0, 1], ['no fight', 'fight']); plt.yticks([0, 1], ['no fight', 'fight'])
for i in range(2):
    for j in range(2): plt.text(j, i, cm[i, j], ha='center', va='center')
plt.xlabel('predicted'); plt.ylabel('true'); plt.title('Validation set'); os.makedirs(f'{DATA}/results', exist_ok=True)
plt.savefig(f'{DATA}/results/confusion_matrix.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Which val clips are wrong? Open them on Drive and look: false alarms tell us which hard negatives are missing.
import pandas as pd
wrong = [{'score': round(float(s), 3), 'true': int(y), 'clip': p} for (p, y, _), s in zip(val_items, ps) if (s >= 0.5) != bool(y)]
wrong_df = pd.DataFrame(wrong, columns=['score', 'true', 'clip']).sort_values('score', ascending=False)
wrong_df.to_csv(f'{DATA}/results/val_mistakes.csv', index=False)
print(len(wrong_df), 'mistakes (true=0: false alarm, true=1: missed)')
wrong_df

## 5. Test on our own staged clips (the numbers that matter)
The validation set tells us the model works on adult fights. This section tells us whether it works **for us**.
Look at the **hug**, **high five** and **normal play** rows: they must stay low. The push row must be high.

In [ ]:
import pandas as pd
model.eval(); rows = []
for p, y, scene in staged_items:
    feats = torch.from_numpy(cached_features(p)).float().unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        rows.append({'scene': scene, 'clip': os.path.basename(p), 'violent': y,
                     'score': torch.sigmoid(model(feats)).item()})
staged_df = pd.DataFrame(rows, columns=['scene', 'clip', 'violent', 'score'])
if staged_df.empty:
    print('No staged clips yet. Film them, put them in', STAGED_DIR, 'and re-run from section 1.')
else:
    print(staged_df.groupby('scene')['score'].agg(['count', 'mean', 'max']).round(3))
staged_df.sort_values('score', ascending=False).head(20)

In [ ]:
# Pick the two thresholds from staged clips, not from RWF.
#   NORMAL: few false alarms (hug / play stay below it).
#   LOW   : used when a child is in the pair; accepts more false alarms for recall.
def stats(th):
    pred = staged_df['score'] >= th
    tp = int((pred & (staged_df.violent == 1)).sum()); fp = int((pred & (staged_df.violent == 0)).sum())
    fn = int((~pred & (staged_df.violent == 1)).sum())
    return th, tp, fp, fn
print(pd.DataFrame([stats(t) for t in np.arange(0.1, 0.95, 0.05)], columns=['threshold', 'TP', 'FP', 'FN']).round(2))

In [ ]:
NORMAL_THRESHOLD = 0.70   # set from the table above
LOW_THRESHOLD    = 0.45   # child involved; check hugs still stay below this
print('hugs / high fives above LOW:')
print(staged_df[(staged_df.violent == 0) & (staged_df.score >= LOW_THRESHOLD)])

In [ ]:
# Save every result that matters to Drive (survives the Colab session). Run after section 5.
os.makedirs(f'{DATA}/results', exist_ok=True)
ps, ys = predict(val_dl); pred = ps > 0.5
metrics = {
    'val_accuracy': float(accuracy_score(ys, pred)), 'val_precision': float(precision_score(ys, pred, zero_division=0)),
    'val_recall': float(recall_score(ys, pred, zero_division=0)), 'val_f1': float(f1_score(ys, pred, zero_division=0)),
    'val_confusion_matrix': confusion_matrix(ys, pred).tolist(),
    'n_train': len(train_items), 'n_val': len(val_items), 'n_staged': len(staged_items),
    'normal_threshold': NORMAL_THRESHOLD, 'low_threshold': LOW_THRESHOLD,
    'history': history,
}
json.dump(metrics, open(f'{DATA}/results/metrics.json', 'w'), indent=2)
staged_df.to_csv(f'{DATA}/results/staged_scores.csv', index=False)
print(json.dumps(metrics, indent=2))
print('saved to', f'{DATA}/results')

## 6. Export
`violence.pt` holds only the small LSTM head plus its settings. The backbone is rebuilt from torchvision ImageNet weights at load time.
Share it through Drive or a Hugging Face model repo. Do not commit it.

In [ ]:
torch.save({
    'head_state': model.state_dict(),
    'seq_len': SEQ_LEN, 'img_size': IMG, 'feat_dim': FEAT_DIM, 'hidden': 256,
    'normal_threshold': NORMAL_THRESHOLD, 'low_threshold': LOW_THRESHOLD,
    'mean': MEAN.tolist(), 'std': STD.tolist(),
}, OUT_PATH)
print('saved', OUT_PATH, os.path.getsize(OUT_PATH) // 1024, 'KB')